In [1]:
import pandas as pd
import json
import re
import os
from pathlib import Path

In [2]:
df = pd.read_csv("explanations.csv")

In [9]:
items = df.iloc[87:90]['explainer_3'].tolist()
for item in items:
    print(item + "\n")

CONTEXT: There are 5 houses, numbered 1 to 5 from left to right, as seen from across the street. Each house is occupied by a different person. Each house has a unique attribute for each of the following characteristics:  - Each person has a unique name: `Peter`, `Alice`, `Arnold`, `Bob`, `Eric`  - Each person has a unique hobby: `photography`, `cooking`, `knitting`, `gardening`, `painting`  - Each person has a unique favorite drink: `root beer`, `milk`, `water`, `coffee`, `tea`  ## Clues: 1. Eric is the coffee drinker. 2. The tea drinker is the person who paints as a hobby. 3. The person who enjoys knitting is not in the fourth house. 4. Peter is not in the fourth house. 5. Eric is somewhere to the right of the root beer lover. 6. Arnold is the person who loves cooking. 7. The one who only drinks water is somewhere to the right of the person who enjoys gardening. 8. There is one house between Bob and the person who paints as a hobby. 9. The person who enjoys gardening is directly left 

In [2]:
DEFAULT_CONTROL_QUESTION = {
    "question": "What was the question about?",
    "correctAnswer": "Correct answer",
    "wrongAnswers": [
        "Wrong Answer",
        "Wrong Answer",
        "Wrong Answer",
        "Wrong Answer",
    ],
}
FEVER_LABEL_MAP = {
    "supported": True,
    "supports": True,
    "refuted": False,
    "refutes": False,
    "not enough evidence": None,
    "not enough info": None,
    "conflicting evidence/cherry-picking": None,
    "conflicting evidence": None,
    "cherry-picking": None,
}

FEVER_LABELS = [
    "Supported",
    "Refuted",
    "Not Enough Evidence",
    "Conflicting Evidence/Cherry-picking",
]

def capitalize_sentences(text):
    text = text.strip()
    if not text:
        return text
    text = text[0].upper() + text[1:]
    # capitalize next letter after whitespace
    text = re.sub(r"([.!?])([ \t]+)([a-z])", lambda m: m.group(1) + m.group(2) + m.group(3).upper(), text)
    # capitalize next letter after newlines
    text = re.sub(r"(\n+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)
    return text

def parse_boolq_task(task):
    task = task.strip()

    q_match = re.search(r"Question:\s*(.*?)(?=\n\s*\n|\n\s*Passage:|\Z)", task, re.DOTALL | re.IGNORECASE)
    question_raw = q_match.group(1).strip() if q_match else ""
    question = capitalize_sentences(question_raw)
    if question and not question.endswith("?"):
        question += "?"

    p_match = re.search(r"Passage:\s*(.*?)(?=\n\s*\n\s*Answer the question|\n\s*Answer the question|\Z)",task, re.DOTALL | re.IGNORECASE)
    passage_raw = p_match.group(1).strip() if p_match else ""
    passage = capitalize_sentences(passage_raw)

    return question, passage

def parse_fever_task(task):
    task = task.strip()

    claim_match = re.search(r"Claim:\s*(.*?)(?=\n\s*\n|\n\s*Evidence:|\Z)", task, re.DOTALL | re.IGNORECASE)
    claim_raw = claim_match.group(1).strip() if claim_match else ""
    claim_raw = re.sub(r'^[\"\u201c\u2018]+|[\"\u201d\u2019]+$', "", claim_raw).strip()
    claim = capitalize_sentences(claim_raw)

    evidence_match = re.search(r"Evidence:\s*(.*?)(?=\n\s*\n\s*Choose\s+which|\n\s*Choose\s+which|\Z)", task, re.DOTALL | re.IGNORECASE)
    evidence_raw = evidence_match.group(1).strip() if evidence_match else ""

    # Split evidence into Q&A pairs
    evidence = []
    sentences = re.split(r'(?<=[.?])\s+', evidence_raw.strip())
    i = 0
    while i < len(sentences):
        s = sentences[i].strip()
        if not s:
            i += 1
            continue
        evidence.append(capitalize_sentences(s))
        i += 1

    labels_match = re.search(r"Choose\s+which\b.*?:\s*\n(.*?)(?=\Z)", task, re.DOTALL | re.IGNORECASE)
    if labels_match:
        raw_lines = [l.strip() for l in labels_match.group(1).splitlines() if l.strip()]
        label_options = raw_lines if raw_lines else FEVER_LABELS
    else:
        label_options = FEVER_LABELS

    return claim, evidence, label_options

def ensure_bold_sentences(text):
    if "**" in text:
        return text

    sentences = re.split(r'(?<=\.)\s+', text.strip())
    bolded = []
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            if not sentence.endswith('.'):
                sentence += '.'
            bolded.append(f"**{sentence}**")

    return " ".join(bolded)

def bold_to_sentiment(text):
    return re.sub(
        r"\*\*(.+?)\*\*",
        lambda m: f"<mark>{m.group(1).strip()}</mark>",
        text,
        flags=re.DOTALL,
    )

def bold_to_sentiment_merged(attribution_text, full_content):
    if len(attribution_text)>len(full_content):
        return bold_to_sentiment(attribution_text)
    raw_sentences = re.split(r'(?<=\.)\s+', attribution_text.strip())
    result = full_content

    for sentence in raw_sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        plain = re.sub(r'\*\*(.+?)\*\*', r'\1', sentence, flags=re.DOTALL).strip()
        marked = re.sub(r'\*\*(.+?)\*\*', lambda m: f"<mark>{m.group(1).strip()}</mark>", sentence, flags=re.DOTALL)
        if plain and plain in result:
            result = result.replace(plain, marked, 1)
    return result

def fever_label_to_bool(label: str):
    return FEVER_LABEL_MAP.get(label.strip().lower(), None)

def parse_zebralogic_task(task):
    # CONTEXT — split into text and clues
    ctx_m = re.search(r"CONTEXT:\s*(.*?)(?=\n\s*QUESTION:|\Z)", task, re.DOTALL | re.IGNORECASE)
    context_raw = ctx_m.group(1).strip() if ctx_m else task.strip()
    context_raw = re.sub(r"`([^`]+)`", r"\1", context_raw)
    clues_split = re.split(r'\n\s*#+\s*Clues\s*:\s*\n|\n\s*Clues\s*:\s*\n', context_raw, flags=re.IGNORECASE)
    context_text = capitalize_sentences(clues_split[0].strip())
    clues = []
    if len(clues_split) > 1:
        for line in clues_split[1].splitlines():
            m = re.match(r'^\d+\.\s+(.+)$', line.strip())
            if m:
                clues.append(capitalize_sentences(m.group(1).strip()))

    # QUESTION
    q_m = re.search(r"QUESTION:\s*(.*?)(?=\n\s*OPTIONS:|\Z)", task, re.DOTALL | re.IGNORECASE)
    question = capitalize_sentences(q_m.group(1).strip()) if q_m else ""

    # OPTIONS
    opt_m = re.search(r"OPTIONS:\s*(.*?)(?=\Z)", task, re.DOTALL | re.IGNORECASE)
    options = []
    if opt_m:
        for line in opt_m.group(1).splitlines():
            m = re.match(r"^[A-Z][).]\s*(.+)$", line.strip())
            if m:
                t = m.group(1).strip()
                options.append((t[0].upper() + t[1:]) if t else t)

    return context_text, clues, question, options

def letter_to_option(letter, options):
    idx = ord(letter.upper()) - ord("A")
    return options[idx] if 0 <= idx < len(options) else letter

In [3]:
def build_dataset_items(items_df, explanations_df, dataset_name):
    subset = explanations_df[explanations_df["dataset"].str.lower() == dataset_name.lower()].drop(columns=['explainer_1', 'explainer_2']).copy()

    items = []
    for _idx, group in subset.groupby("original_index", sort=True):
        anchor = items_df[items_df['original_index']==_idx]
        label = anchor['label'].iloc[0]
        pred = anchor['predicted_label'].iloc[0]
        # Extract the three explanations per og_idx
        xai_map = {}
        for _, row in group.iterrows():
            xai_map[str(row["xai_type"]).lower().strip()] = row
        xai_features = {"truthfulness": None}

        # XAI feature content usually stored in explainer_3
        task = xai_map["counterfactual"]["explainer_3"]
        p_match = re.search(
            r"Passage:\s*(.*?)(?=\s*Answer\s+the\s+question|\Z)",
            task, re.DOTALL | re.IGNORECASE
        )
        passage_raw = p_match.group(1).strip() if p_match else task
        passage = capitalize_sentences(passage_raw)
        xai_features["counterfactualExplanation"]  = passage
        xai_features["naturalLanguageExplanation"] = xai_map["rationale"]["explainer_3"]

        # Shared fields
        base = {
            "id":              _idx,
            "isFalsePositive": False,
            "isTrueNegative":  False,
            "isQualification": False, 
            "dataset":        dataset_name,
            "xaiFeatures":     xai_features,
            "controlQuestion": DEFAULT_CONTROL_QUESTION,
        }
        # Dataset-specific fields
        task_text = xai_map["attribution"]["task"]
        if dataset_name.lower() == "zebralogic":
            context, clues, question, options = parse_zebralogic_task(task_text)
            xai_features['truthfulness'] = options[label]
            item = {**base, "title": question, "content": context, "clues": clues, "ratingType": "multiple-choice", "options": options}
            xai_features["highlightedContent"] = bold_to_sentiment_merged(xai_map["attribution"]["explainer_3"], context)

        elif dataset_name.lower() == "boolq":
            explanation = ensure_bold_sentences(xai_map["attribution"]["explainer_3"])
            question, passage = parse_boolq_task(task_text)
            xai_features['truthfulness'] = bool(label)
            item = {**base, "title": question, "content": passage, "ratingType": "boolean"}
            xai_features["highlightedContent"] = bold_to_sentiment_merged(explanation, passage)

            # print(xai_map["attribution"]["explainer_3"])
            # print("======================================================")
            # print(explanation)
            # print("======================================================")
            # print(bold_to_sentiment_merged(explanation, passage))
        
        elif dataset_name.lower() == "fever":
            claim, evidence, options = parse_fever_task(task_text)
            xai_features['truthfulness'] = options[label]
            # Flatten evidence to plain text for highlight merging
            evidence_text = " ".join(e for e in evidence)
            item = {**base, "claim": claim, "evidence": evidence, "ratingType": "multiple-choice", "options": options}
            xai_features["highlightedContent"] = bold_to_sentiment_merged(xai_map["attribution"]["explainer_3"], claim + '\n' + evidence_text)
        items.append(item)

    return items


def generate_dataset_jsons(item_csv_path, ex_sv_path, output_dir = ".", datasets = ("boolq", "zebralogic", "fever")):
    os.makedirs(output_dir, exist_ok=True)
    explanations_df = pd.read_csv(ex_sv_path)
    items_df = pd.read_csv(item_csv_path)

    results = {}
    for name in datasets:
        items = build_dataset_items(items_df, explanations_df, name)
        out_path = os.path.join(output_dir, f"{name}-items.json")
        with open(out_path, "w", encoding="utf-8") as fh:
            json.dump(items, fh, indent=2, ensure_ascii=False)
        results[name] = items

    return results


In [5]:
results = generate_dataset_jsons(item_csv_path='dataset-items.csv', ex_sv_path="explanations.csv", output_dir="./data/")